# Few-Shot Prompt with Templates

The `few_shot_with_templates.py` module defines a few-shot string prompt whose prefix and suffix are prompt-template objects. It supports fixed examples or dynamically selected examples and provides synchronous and asynchronous formatting.

# FewShotPromptWithTemplates: `StringPromptTemplate`

`FewShotPromptWithTemplates` creates a few-shot prompt by combining an optional prefix template, formatted examples, and a required suffix template. Each example is formatted using an `example_prompt`, and the combined sections are processed using the selected template format.

   **Syntax**

   ```python
      FewShotPromptWithTemplates(
      self,
      *args: Any = (),
      **kwargs: Any = {}
      )
   ```


## Fields

1. `examples`:`list[dict[str, Any]] | None`:= Stores a fixed list of example dictionaries used to construct the prompt. Its default value is `None`.

2. `example_selector`:`BaseExampleSelector | None`:= Stores an optional selector that dynamically chooses examples using the supplied prompt inputs. Its default value is `None`.

3. `example_prompt`:`PromptTemplate`:= Stores the prompt template used to format each individual example.

4. `suffix`:`StringPromptTemplate`:= Stores the string prompt template placed after the formatted examples.

5. `example_separator`:`str`:= Stores the separator used to join the prefix, formatted examples, and suffix. Its default value is `"\n\n"`.

6. `prefix`:`StringPromptTemplate | None`:= Stores an optional string prompt template placed before the formatted examples. Its default value is `None`.

7. `template_format`:`PromptTemplateFormat`:= Specifies the formatting syntax used for the combined prompt. Its default value is `"f-string"`. Supported values are `"f-string"`, `"jinja2"`, and `"mustache"`.

8. `validate_template`:`bool`:= Determines whether the prefix, suffix, partial variables, and declared input variables are validated for consistency. Its default value is `False`.

## Configuration

1. `model_config`:`ConfigDict`:= Allows arbitrary Python types and rejects undeclared fields.

   **Syntax**

   ```python
   model_config = ConfigDict(
       arbitrary_types_allowed=True, # Allow arbitrary Python types
       extra="forbid" # Reject undeclared fields
   )
   ```

In [1]:
from langchain_core.prompts import FewShotPromptWithTemplates, PromptTemplate

examples = [
    {"word": "happy", "opposite": "sad"},
    {"word": "hot", "opposite": "cold"}
]

example_prompt = PromptTemplate(
    input_variables=["word", "opposite"],
    template="Word: {word}\nOpposite: {opposite}"
)

prefix = PromptTemplate(
    input_variables=[],
    template="Provide the opposite of each word."
)

suffix = PromptTemplate(
    input_variables=["input_word"],
    template="Word: {input_word}\nOpposite:"
)

prompt = FewShotPromptWithTemplates(
    examples=examples, # Fixed examples used in the prompt
    example_selector=None, # No dynamic example selector
    example_prompt=example_prompt, # Template used to format each example
    suffix=suffix, # Template placed after the examples
    example_separator="\n---\n", # Separator between prompt sections
    prefix=prefix, # Template placed before the examples
    template_format="f-string", # Formatting syntax
    validate_template=True, # Validate the declared variables
    input_variables=["input_word"] # Variables required while formatting
)

result = prompt.format(
    input_word="fast"
)

print(result)

Provide the opposite of each word.
---
Word: happy
Opposite: sad
---
Word: hot
Opposite: cold
---
Word: fast
Opposite:


## Validators

1. `check_examples_and_selector`:= Validates that exactly one of `examples` or `example_selector` is provided.

   This validator runs automatically while the prompt template is being created.

   **Syntax**

   ```python
   @classmethod
   check_examples_and_selector(
       cls, # FewShotPromptWithTemplates class
       values: dict[str, Any] # Values supplied while creating the prompt template
   ) -> Any
   ```

2. `template_is_valid`:= Validates the declared input variables against the prefix, suffix, and partial variables.

   When template validation is disabled, the required input variables are derived automatically. This validator runs automatically after the prompt template is created.

   **Syntax**

   ```python
   template_is_valid(
       self # Prompt template instance to validate
   ) -> Self
   ```


In [2]:
from langchain_core.prompts import FewShotPromptWithTemplates, PromptTemplate

examples = [
    {"word": "happy", "opposite": "sad"},
    {"word": "hot", "opposite": "cold"}
]

example_prompt = PromptTemplate(
    input_variables=["word", "opposite"],
    template="Word: {word}\nOpposite: {opposite}"
)

prefix = PromptTemplate(
    input_variables=[],
    template="Provide the opposite of each word."
)

suffix = PromptTemplate(
    input_variables=["input_word"],
    template="Word: {input_word}\nOpposite:"
)

# Valid: exactly one of examples or example_selector is provided
# and input_variables matches the variables required by the templates.
prompt = FewShotPromptWithTemplates(
    examples=examples,
    example_selector=None,
    example_prompt=example_prompt,
    prefix=prefix,
    suffix=suffix,
    input_variables=["input_word"],
    validate_template=True
)

print(prompt.format(input_word="fast"))

# Invalid: neither examples nor example_selector is provided.
try:
    FewShotPromptWithTemplates(
        example_prompt=example_prompt,
        prefix=prefix,
        suffix=suffix
    )
except ValueError as error:
    print(error)

# Invalid: the declared input variable does not match the suffix variable.
try:
    FewShotPromptWithTemplates(
        examples=examples,
        example_prompt=example_prompt,
        prefix=prefix,
        suffix=suffix,
        input_variables=["wrong_variable"],
        validate_template=True
    )
except ValueError as error:
    print(error)

Provide the opposite of each word.

Word: happy
Opposite: sad

Word: hot
Opposite: cold

Word: fast
Opposite:
1 validation error for FewShotPromptWithTemplates
  Value error, One of 'examples' and 'example_selector' should be provided [type=value_error, input_value={'example_prompt': Prompt...nput_word}\nOpposite:')}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
1 validation error for FewShotPromptWithTemplates
  Value error, Got input_variables=['wrong_variable'], but based on prefix/suffix expected {'input_word'} [type=value_error, input_value={'examples': [{'word': 'h...alidate_template': True}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


## Methods

1. `get_lc_namespace`:= Returns the LangChain namespace assigned to this prompt-template class.

   **Syntax**

   ```python
   @classmethod
   get_lc_namespace(
       cls # FewShotPromptWithTemplates class
   ) -> list[str]
   ```

2. `format`:= Synchronously retrieves fixed examples or selects examples dynamically, formats each prompt section, combines the sections, and returns the final string.

   **Syntax**

   ```python
   format(
       self, # Few-shot prompt template instance
       **kwargs: Any # Variables used for example selection and prompt formatting
   ) -> str
   ```

3. `aformat`:= Asynchronously retrieves fixed examples or selects examples dynamically, formats each prompt section, combines the sections, and returns the final string.

   **Syntax**

   ```python
   async aformat(
       self, # Few-shot prompt template instance
       **kwargs: Any # Variables used for example selection and prompt formatting
   ) -> str
   ```

4. `save`:= Saves the prompt template to a file when fixed examples are used.

   Saving a prompt that uses an `example_selector` is not supported. This method is deprecated in favour of the LangChain load and dump utilities.

   **Syntax**

   ```python
   save(
       self, # Few-shot prompt template instance
       file_path: Path | str # Destination path for the serialized prompt
   ) -> None
   ```


In [3]:
from langchain_core.prompts import FewShotPromptWithTemplates, PromptTemplate

examples = [
    {"word": "happy", "opposite": "sad"},
    {"word": "hot", "opposite": "cold"}
]

example_prompt = PromptTemplate(
    template="Word: {word}\nOpposite: {opposite}",
    input_variables=["word", "opposite"]
)

prefix = PromptTemplate.from_template(
    "Provide the opposite of each word."
)

suffix = PromptTemplate.from_template(
    "Word: {input_word}\nOpposite:"
)

prompt = FewShotPromptWithTemplates(
    examples=examples,
    example_prompt=example_prompt,
    prefix=prefix,
    suffix=suffix,
    input_variables=["input_word"]
)

# Display the LangChain serialization namespace
print(prompt.get_lc_namespace())

# Format the prompt synchronously
print(prompt.format(input_word="fast"))

# Format the prompt asynchronously in Jupyter Notebook
print(await prompt.aformat(input_word="large"))

# Save the prompt containing fixed examples
prompt.save("few_shot_prompt.json")

['langchain', 'prompts', 'few_shot_with_templates']
Provide the opposite of each word.

Word: happy
Opposite: sad

Word: hot
Opposite: cold

Word: fast
Opposite:
Provide the opposite of each word.

Word: happy
Opposite: sad

Word: hot
Opposite: cold

Word: large
Opposite:


C:\Users\asish\AppData\Local\Temp\ipykernel_13480\2039688651.py:39: LangChainDeprecationWarning: The method `FewShotPromptWithTemplates.save` was deprecated in langchain-core 1.2.21 and will be removed in 2.0.0. Use `Use `dumpd`/`dumps` from `langchain_core.load` to serialize prompts and `load`/`loads` to deserialize them.` instead.
  prompt.save("few_shot_prompt.json")
